# KSC RSAM source-steered launch detector development notebook

This notebook replaces the earlier “compute launch metrics over whole launch windows” workflow with a detection-first workflow:

1. load launch windows, launchpad coordinates, StationXML inventory, and SDS waveform archive
2. read long launch windows without instrument correction
3. compute 1-second RSAM from raw counts
4. compute airwave travel times from each station to one or more launchpads
5. shift RSAM traces into estimated source/origin time and stack them
6. run `flovopy.processing.detection` tools on the stacked traces
7. save intermediate products so every step can be inspected

The default run is limited to the two events that have already loaded successfully. Increase `test_event_indices` or set it to `None` to scale up.

## 1. Configuration

Edit this cell first. Keep the run small while developing; all outputs are written under `outdir / "rsam_stack_detector"`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import re
import fnmatch
import json
import importlib.util
from typing import Any, Dict, Iterable, Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import UTCDateTime, read_inventory, Stream, Trace
from obspy.geodetics.base import gps2dist_azimuth

# FLOVOpy imports. Prefer installed package versions in your working environment.
from flovopy.enhanced.sdsclient import EnhancedSDSClient

try:
    from flovopy.processing.sam import RSAM
except Exception:
    # Development fallback: local sam.py beside this notebook or in /mnt/data.
    for candidate in [Path("sam.py"), Path("/mnt/data/sam.py")]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("sam_local", candidate)
            sam_local = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(sam_local)
            RSAM = sam_local.RSAM
            break
    else:
        raise

try:
    from flovopy.processing.detection import run_coincidence_trigger_dataframe, detect_network_event
except Exception:
    # Development fallback: attached detection module.
    for candidate in [Path("detection.py"), Path("detection(6).py"), Path("/mnt/data/detection(6).py")]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("detection_local", candidate)
            detection_local = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(detection_local)
            run_coincidence_trigger_dataframe = detection_local.run_coincidence_trigger_dataframe
            detect_network_event = detection_local.detect_network_event
            break
    else:
        raise

# -----------------------------
# User-editable paths
# -----------------------------
events_csv = Path("all_florida_launches_with_seed_ids.csv")
launchpads_csv = Path("launchpads.csv")
stationxml = Path("/Volumes/haldata/KSC/station_metadata/KSC.xml")
sds_root = Path("/Volumes/haldata/remastered/SDS_KSC")

outdir = Path("~/work/KSC_ensemble").expanduser() / "rsam_stack_detector"
outdir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Run limits
# -----------------------------
# These correspond to the two events that previously produced usable data.
# Set to None to process all rows after the workflow is validated.
test_event_indices = [1, 2]

# Optional date filter applied before test_event_indices.
date_from = None      # e.g. "2016-01-01"
date_to = None        # e.g. "2017-01-01"
max_events = None     # optional after date filtering and before test_event_indices

# -----------------------------
# Waveform selection
# -----------------------------
networks = ["*"]
stations = ["*"]
locations = ["00", ""]
channels = ["DD?", "DH?"]

# -----------------------------
# RSAM / stacking settings
# -----------------------------
rsam_sampling_interval_s = 1.0
rsam_filter = [0.5, 20.0]
rsam_metric = "mean"       # RSAM dataframe column: mean, median, max, rms, etc.
stack_velocity_mps = 340.0
stack_bin_s = 1.0
min_stack_channels = 1
normalize_before_stack = True

# If a launchpad column is present in the events CSV, it will be used.
# If not present or not matched, all pads in launchpads.csv are stacked.
pad_column_candidates = [
    "pad", "launchpad", "launch_pad", "slc", "SLC", "launch_complex",
    "launch_site", "site", "complex",
]

# -----------------------------
# Detection settings for stacked RSAM traces
# -----------------------------
detection_sta_s = 5.0
detection_lta_s = 60.0
detection_threshold_on = 3.0
detection_threshold_off = 1.2
# One stack trace per pad; min_channels=1 is appropriate for running detector on stack streams.
detection_min_channels = 1
detection_max_events_per_stack = 20

print(f"Output directory: {outdir}")

## 2. Utility functions

In [ ]:
def parse_utc(value) -> Optional[pd.Timestamp]:
    if value is None or value == "":
        return None
    return pd.to_datetime(value, utc=True, errors="coerce")


def ensure_utc(value) -> UTCDateTime:
    if isinstance(value, UTCDateTime):
        return value
    ts = pd.to_datetime(value, utc=True, errors="coerce")
    if pd.isna(ts):
        raise ValueError(f"Cannot parse UTC time: {value!r}")
    return UTCDateTime(ts.to_pydatetime())


def safe_slug(s: Any, maxlen: int = 96) -> str:
    s = str(s) if s is not None else ""
    s = s.strip().replace("/", "-")
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return (s[:maxlen] or "event")


def match_any(value: str, patterns: Iterable[str]) -> bool:
    patterns = list(patterns) if patterns is not None else ["*"]
    return any(fnmatch.fnmatch(str(value), pat) for pat in patterns)


def filter_stream_patterns(
    st: Stream,
    network_patterns=("*",),
    station_patterns=("*",),
    location_patterns=("*",),
    channel_patterns=("*",),
) -> Stream:
    out = Stream()
    for tr in st:
        if not match_any(tr.stats.network, network_patterns):
            continue
        if not match_any(tr.stats.station, station_patterns):
            continue
        if not match_any(tr.stats.location or "", location_patterns):
            continue
        if not match_any(tr.stats.channel, channel_patterns):
            continue
        out += tr
    return out


def normalize_pad_name(x: Any) -> Optional[str]:
    if x is None or pd.isna(x):
        return None
    s = str(x).upper().strip()
    # Strip common textual clutter but keep LC/LZ identifiers.
    s = s.replace("CAPE CANAVERAL", "")
    s = s.replace("CCSFS", "")
    s = s.replace("KSC", "")
    s = s.replace("LAUNCH COMPLEX", "LC")
    s = s.replace("SPACE LAUNCH COMPLEX", "SLC")
    s = s.replace("SLC-", "LC")
    s = s.replace("SLC ", "LC")
    s = s.replace("LC-", "LC")
    s = s.replace("LZ-", "LZ")
    s = s.replace(" ", "")
    s = s.replace("_", "")
    s = s.replace(".", "")
    # Pull a known-looking pad token out of longer strings.
    m = re.search(r"(LC39A|LC39B|LC40|LC41|LC37B|LC46|LZ1|LZ2)", s)
    if m:
        return m.group(1)
    return s or None


def choose_pad_column(row: pd.Series, candidates: Sequence[str]) -> tuple[Optional[str], Optional[str]]:
    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            norm = normalize_pad_name(row[col])
            if norm:
                return col, norm
    return None, None


def event_identifier(row: pd.Series, fallback_index: Optional[int] = None) -> str:
    for col in ["event_id", "seed_id", "launch_id", "id"]:
        if col in row.index and pd.notna(row[col]):
            return safe_slug(row[col])
    if "window_start" in row.index and pd.notna(row["window_start"]):
        return safe_slug(pd.to_datetime(row["window_start"], utc=True).strftime("%Y%m%dT%H%M%SZ"))
    return f"event_{fallback_index:04d}" if fallback_index is not None else "event"


def event_slug(row: pd.Series, fallback: str) -> str:
    for col in ["slug", "mission", "name", "payload", "vehicle"]:
        if col in row.index and pd.notna(row[col]):
            return safe_slug(row[col])
    return fallback

## 3. Load launch windows, launchpads, StationXML, and SDS

In [ ]:
# Resolve files. This fallback is useful when testing this notebook from the ChatGPT sandbox.
if not events_csv.exists() and Path("/mnt/data/all_florida_launches_with_seed_ids.csv").exists():
    events_csv = Path("/mnt/data/all_florida_launches_with_seed_ids.csv")
if not launchpads_csv.exists() and Path("/mnt/data/launchpads.csv").exists():
    launchpads_csv = Path("/mnt/data/launchpads.csv")

if not events_csv.exists():
    raise FileNotFoundError(f"events_csv not found: {events_csv.resolve()}")
if not launchpads_csv.exists():
    raise FileNotFoundError(f"launchpads_csv not found: {launchpads_csv.resolve()}")
if not stationxml.exists():
    raise FileNotFoundError(f"StationXML not found: {stationxml}")
if not sds_root.exists():
    raise FileNotFoundError(f"SDS root not found: {sds_root}")

launch_df = pd.read_csv(events_csv)
required = {"window_start", "window_end"}
missing = sorted(required - set(launch_df.columns))
if missing:
    raise ValueError(f"events CSV is missing required columns: {missing}")

launch_df["window_start"] = pd.to_datetime(launch_df["window_start"], utc=True, errors="coerce")
launch_df["window_end"] = pd.to_datetime(launch_df["window_end"], utc=True, errors="coerce")
launch_df["window_duration_s"] = (launch_df["window_end"] - launch_df["window_start"]).dt.total_seconds()
launch_df = launch_df.dropna(subset=["window_start", "window_end"]).copy()

if date_from is not None:
    launch_df = launch_df[launch_df["window_end"] >= parse_utc(date_from)].copy()
if date_to is not None:
    launch_df = launch_df[launch_df["window_start"] <= parse_utc(date_to)].copy()
if max_events is not None:
    launch_df = launch_df.head(int(max_events)).copy()

if test_event_indices is not None:
    event_df = launch_df.iloc[test_event_indices].copy()
else:
    event_df = launch_df.copy()

event_df = event_df.reset_index(drop=False).rename(columns={"index": "catalog_row"})

launchpads_df = pd.read_csv(launchpads_csv)
launchpads_df.columns = [c.strip() for c in launchpads_df.columns]
if "pad" not in launchpads_df.columns:
    # tolerate common alternate first-column names
    launchpads_df = launchpads_df.rename(columns={launchpads_df.columns[0]: "pad"})
for required_col in ["pad", "lat", "lon"]:
    if required_col not in launchpads_df.columns:
        raise ValueError(f"launchpads CSV must contain {required_col!r}; columns are {launchpads_df.columns.tolist()}")
launchpads_df["pad"] = launchpads_df["pad"].astype(str).str.upper().str.strip()
launchpads_df["pad_norm"] = launchpads_df["pad"].map(normalize_pad_name)

inv = read_inventory(str(stationxml))
sdsobject = EnhancedSDSClient(sds_root)

print(f"Loaded {len(launch_df)} launch windows from {events_csv}")
print(f"Running {len(event_df)} events in this notebook run")
print(f"Loaded {len(launchpads_df)} launchpads from {launchpads_csv}: {launchpads_df['pad'].tolist()}")
print(inv)
display(event_df.head())

## 4. Inventory coordinate lookup

Station coordinates are taken from the StationXML inventory, not trace metadata. The lookup tries exact `NET.STA.LOC.CHA`, then location-insensitive `NET.STA.CHA`, then station-level `NET.STA`.

In [ ]:
def inventory_coord_lookup(inv) -> dict[str, dict[str, float]]:
    coords = {}
    for net in inv:
        for sta in net:
            sta_key = f"{net.code}.{sta.code}"
            coords[sta_key] = {
                "latitude": float(sta.latitude),
                "longitude": float(sta.longitude),
                "elevation": float(sta.elevation or 0.0),
            }
            for cha in sta:
                loc = cha.location_code or ""
                exact = f"{net.code}.{sta.code}.{loc}.{cha.code}"
                no_loc = f"{net.code}.{sta.code}.{cha.code}"
                c = {
                    "latitude": float(cha.latitude if cha.latitude is not None else sta.latitude),
                    "longitude": float(cha.longitude if cha.longitude is not None else sta.longitude),
                    "elevation": float(cha.elevation if cha.elevation is not None else (sta.elevation or 0.0)),
                }
                coords[exact] = c
                coords[no_loc] = c
    return coords


inv_coords = inventory_coord_lookup(inv)
print(f"Inventory coordinate keys: {len(inv_coords)}")
for sta in ["BCHH", "FIRE", "TANK"]:
    print(sta, [k for k in inv_coords if f".{sta}" in k][:8])


def get_trace_lat_lon_from_inventory(tr: Trace, inv_coords: dict[str, dict[str, float]]) -> tuple[float, float]:
    net = tr.stats.network
    sta = tr.stats.station
    loc = tr.stats.location or ""
    cha = tr.stats.channel
    keys = [
        f"{net}.{sta}.{loc}.{cha}",
        f"{net}.{sta}.{cha}",
        f"{net}.{sta}",
    ]
    for key in keys:
        if key in inv_coords:
            c = inv_coords[key]
            return c["latitude"], c["longitude"]
    raise KeyError(f"No inventory coordinates for {tr.id}; tried {keys}")

## 5. RSAM and source-steered stack functions

In [ ]:
def read_event_stream(row: pd.Series, sds: EnhancedSDSClient) -> Stream:
    t0 = ensure_utc(row["window_start"])
    t1 = ensure_utc(row["window_end"])
    st = sds.read(
        starttime=t0,
        endtime=t1,
        net=networks,
        sta=stations,
        loc="*",
        chan=channels,
        skip_low_rate_channels=True,
        merge=None,
        trim=True,
        daywise=False,
        postprocess=False,
        final_smart_merge=False,
        verbose=False,
    )
    if st is None:
        return Stream()
    st = filter_stream_patterns(st, networks, stations, locations, channels)
    st.trim(t0, t1, pad=False)
    return st


def compute_rsam_1s(st: Stream, freqmin: float, freqmax: float, metric: str = "mean") -> dict[str, pd.DataFrame]:
    """Compute 1-second RSAM from raw counts using the existing RSAM class."""
    rsam = RSAM(
        stream=st.copy(),
        sampling_interval=rsam_sampling_interval_s,
        filter=[float(freqmin), float(freqmax)],
        bands=None,
        corners=4,
        despike=False,
        verbose=False,
    )
    frames = {}
    for seed_id, df in rsam.dataframes.items():
        if metric not in df.columns:
            print(f"Skipping {seed_id}: RSAM metric {metric!r} not in columns {df.columns.tolist()}")
            continue
        d = df[["time", metric]].copy()
        d = d.rename(columns={metric: "value"})
        d["seed_id"] = seed_id
        frames[seed_id] = d
    return frames


def robust_normalize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    scale = 1.4826 * mad if np.isfinite(mad) and mad > 0 else np.nanstd(x)
    if not np.isfinite(scale) or scale == 0:
        return np.full_like(x, np.nan, dtype=float)
    return (x - med) / scale


def write_rsam_frames(rsam_frames: dict[str, pd.DataFrame], event_dir: Path) -> Path:
    rows = []
    for seed_id, df in rsam_frames.items():
        d = df.copy()
        d["seed_id"] = seed_id
        rows.append(d)
    out = event_dir / "rsam_1s_long.csv"
    if rows:
        pd.concat(rows, ignore_index=True).to_csv(out, index=False)
    else:
        pd.DataFrame(columns=["time", "value", "seed_id"]).to_csv(out, index=False)
    return out


def pads_for_event(row: pd.Series, launchpads_df: pd.DataFrame) -> tuple[pd.DataFrame, Optional[str], Optional[str]]:
    col, norm_pad = choose_pad_column(row, pad_column_candidates)
    if norm_pad and norm_pad in set(launchpads_df["pad_norm"]):
        return launchpads_df[launchpads_df["pad_norm"] == norm_pad].copy(), col, norm_pad
    return launchpads_df.copy(), col, norm_pad


def stack_rsam_for_pad(
    rsam_frames: dict[str, pd.DataFrame],
    st_for_coords: Stream,
    pad_row: pd.Series,
    inv_coords: dict[str, dict[str, float]],
    velocity_mps: float = 340.0,
    bin_s: float = 1.0,
    normalize: bool = True,
) -> tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    pad = pad_row["pad"]
    pad_lat = float(pad_row["lat"])
    pad_lon = float(pad_row["lon"])
    tr_by_id = {tr.id: tr for tr in st_for_coords}
    shifted = []

    for seed_id, df in rsam_frames.items():
        tr = tr_by_id.get(seed_id)
        if tr is None:
            continue
        try:
            sta_lat, sta_lon = get_trace_lat_lon_from_inventory(tr, inv_coords)
        except Exception as exc:
            print(f"Skipping {seed_id}: {exc}")
            continue

        dist_m, az, baz = gps2dist_azimuth(pad_lat, pad_lon, sta_lat, sta_lon)
        tt_s = dist_m / float(velocity_mps)
        d = df.copy()
        d["origin_time"] = d["time"] - tt_s
        d["value_used"] = robust_normalize(d["value"].values) if normalize else d["value"].astype(float).values
        d["pad"] = pad
        d["distance_m"] = dist_m
        d["travel_time_s"] = tt_s
        d["azimuth"] = az
        d["backazimuth"] = baz
        shifted.append(d[["origin_time", "time", "value", "value_used", "seed_id", "pad", "distance_m", "travel_time_s", "azimuth", "backazimuth"]])

    if not shifted:
        return None, None

    all_shifted = pd.concat(shifted, ignore_index=True)
    all_shifted["origin_time_bin"] = (np.round(all_shifted["origin_time"] / bin_s) * bin_s).astype(float)

    stack = (
        all_shifted
        .groupby("origin_time_bin")
        .agg(
            stack_mean=("value_used", "mean"),
            stack_median=("value_used", "median"),
            stack_max=("value_used", "max"),
            n=("value_used", "count"),
            n_seed_ids=("seed_id", "nunique"),
        )
        .reset_index()
        .rename(columns={"origin_time_bin": "time"})
    )
    stack = stack[stack["n_seed_ids"] >= int(min_stack_channels)].copy()
    stack["datetime"] = pd.to_datetime(stack["time"], unit="s", utc=True)
    stack["pad"] = pad
    return stack, all_shifted


def stack_dataframe_to_stream(stack_df: pd.DataFrame, pad: str, event_id: str, value_col: str = "stack_mean") -> Stream:
    if stack_df is None or stack_df.empty:
        return Stream()
    d = stack_df.sort_values("time").copy()
    times = d["time"].to_numpy(dtype=float)
    values = d[value_col].to_numpy(dtype=np.float32)
    if len(times) < 2:
        return Stream()
    dt = float(np.nanmedian(np.diff(times)))
    sr = 1.0 / dt if dt > 0 else 1.0
    # Fill small gaps onto a regular 1-s grid.
    grid = np.arange(times[0], times[-1] + 0.5 * dt, dt)
    y = np.interp(grid, times, np.nan_to_num(values, nan=np.nanmedian(values)))
    tr = Trace(data=y.astype(np.float32))
    tr.stats.network = "RS"
    tr.stats.station = safe_slug(str(pad), 5)
    tr.stats.location = ""
    tr.stats.channel = "SAM"
    tr.stats.starttime = UTCDateTime(grid[0])
    tr.stats.sampling_rate = sr
    tr.stats["event_id"] = event_id
    tr.stats["pad"] = pad
    tr.stats["units"] = "normalized_rsam_stack"
    return Stream([tr])


def plot_stacks(stacks: dict[str, pd.DataFrame], event_dir: Path, event_id: str, launch_time: Optional[UTCDateTime] = None) -> Path:
    fig, ax = plt.subplots(figsize=(14, 5))
    plotted = 0
    for pad, stack in stacks.items():
        if stack is None or stack.empty:
            continue
        ax.plot(stack["datetime"], stack["stack_mean"], lw=1.0, label=pad)
        plotted += 1
    if launch_time is not None:
        ax.axvline(pd.to_datetime(launch_time.datetime, utc=True), color="k", ls="--", lw=1.0, label="metadata launch time")
    ax.set_title(f"Travel-time-corrected 1-s RSAM stacks: {event_id}")
    ax.set_xlabel("Estimated source/origin time")
    ax.set_ylabel("Mean normalized RSAM")
    ax.grid(True, alpha=0.3)
    if plotted:
        ax.legend(ncol=4)
    fig.tight_layout()
    out = event_dir / "rsam_stacks.png"
    fig.savefig(out, dpi=150)
    plt.show()
    return out

## 6. Detection on RSAM stack traces

This wraps the attached/installed detection module. Each stack is converted to a 1-Hz ObsPy `Trace`, then `run_coincidence_trigger_dataframe()` is run. Because each pad stack is already a network stack, `min_channels=1` is intentional here.

In [ ]:
def detect_on_stack_stream(st_stack: Stream, event_dir: Path, pad: str) -> pd.DataFrame:
    if st_stack is None or len(st_stack) == 0:
        return pd.DataFrame()
    try:
        det = run_coincidence_trigger_dataframe(
            st_stack,
            trigger_type="recstalta",
            sta_seconds=detection_sta_s,
            lta_seconds=detection_lta_s,
            threshold_on=detection_threshold_on,
            threshold_off=detection_threshold_off,
            min_channels=detection_min_channels,
            pretrigger_seconds=10.0,
            posttrigger_seconds=20.0,
            write_mseed=False,
            outdir=str(event_dir),
            max_events=detection_max_events_per_stack,
            make_plots=False,
            details=True,
        )
    except Exception as exc:
        print(f"Detection failed for pad {pad}: {type(exc).__name__}: {exc}")
        return pd.DataFrame()
    if det is None or det.empty:
        return pd.DataFrame()
    det = det.copy()
    det["pad"] = pad
    return det


def plot_stack_detections(stack_df: pd.DataFrame, detections_df: pd.DataFrame, event_dir: Path, event_id: str, pad: str) -> Path:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(stack_df["datetime"], stack_df["stack_mean"], lw=1.0)
    if detections_df is not None and not detections_df.empty:
        for _, r in detections_df.iterrows():
            on = pd.to_datetime(r.get("on_time", r.get("time", None)), utc=True, errors="coerce")
            off = pd.to_datetime(r.get("off_time", None), utc=True, errors="coerce")
            if pd.notna(on):
                if pd.notna(off):
                    ax.axvspan(on, off, alpha=0.2)
                else:
                    ax.axvline(on, alpha=0.5)
    ax.set_title(f"RSAM stack detections: {event_id} {pad}")
    ax.set_xlabel("Estimated source/origin time")
    ax.set_ylabel("Mean normalized RSAM")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    out = event_dir / f"detections_{safe_slug(pad)}.png"
    fig.savefig(out, dpi=150)
    plt.show()
    return out

## 7. Process one event and save intermediate products

In [ ]:
def process_one_event(row: pd.Series, sds: EnhancedSDSClient, run_index: int) -> dict[str, Any]:
    event_id = event_identifier(row, run_index)
    slug = event_slug(row, event_id)
    event_dir = outdir / f"{run_index:03d}_{event_id}_{slug}"
    event_dir.mkdir(parents=True, exist_ok=True)

    t0 = ensure_utc(row["window_start"])
    t1 = ensure_utc(row["window_end"])
    print(f"\n{run_index}: {event_id} {slug}")
    print(f"  window: {t0} to {t1} ({t1 - t0:.1f} s)")

    # Save the row metadata as JSON/CSV for provenance.
    row_out = row.copy()
    for k, v in row_out.items():
        if isinstance(v, pd.Timestamp):
            row_out[k] = str(v)
    with open(event_dir / "event_metadata.json", "w") as f:
        json.dump(dict(row_out), f, indent=2, default=str)

    st = read_event_stream(row, sds)
    if len(st) == 0:
        print("  no waveform data")
        return {"event_id": event_id, "status": "no_waveform_data", "event_dir": str(event_dir)}
    print(f"  raw traces: {len(st)}")

    # Save trace inventory for inspection.
    trace_rows = []
    for tr in st:
        rec = {
            "seed_id": tr.id,
            "network": tr.stats.network,
            "station": tr.stats.station,
            "location": tr.stats.location,
            "channel": tr.stats.channel,
            "starttime": str(tr.stats.starttime),
            "endtime": str(tr.stats.endtime),
            "sampling_rate": tr.stats.sampling_rate,
            "npts": tr.stats.npts,
        }
        try:
            lat, lon = get_trace_lat_lon_from_inventory(tr, inv_coords)
            rec["latitude"] = lat
            rec["longitude"] = lon
            rec["has_inventory_coords"] = True
        except Exception:
            rec["latitude"] = np.nan
            rec["longitude"] = np.nan
            rec["has_inventory_coords"] = False
        trace_rows.append(rec)
    trace_df = pd.DataFrame(trace_rows)
    trace_df.to_csv(event_dir / "trace_inventory.csv", index=False)
    display(trace_df)

    rsam_frames = compute_rsam_1s(st, rsam_filter[0], rsam_filter[1], metric=rsam_metric)
    print(f"  RSAM traces: {len(rsam_frames)}")
    rsam_csv = write_rsam_frames(rsam_frames, event_dir)
    print(f"  wrote {rsam_csv}")
    if not rsam_frames:
        return {"event_id": event_id, "status": "no_rsam", "event_dir": str(event_dir)}

    pads_to_try, pad_col, pad_norm = pads_for_event(row, launchpads_df)
    if pad_norm and len(pads_to_try) == 1:
        print(f"  using metadata pad {pad_norm!r} from column {pad_col!r}")
    else:
        print(f"  no matched pad in metadata; stacking all {len(pads_to_try)} pads")
        if pad_col:
            print(f"  unmatched metadata pad candidate from {pad_col!r}: {pad_norm!r}")

    stacks = {}
    detections = []
    stack_stream = Stream()

    for _, pad_row in pads_to_try.iterrows():
        pad = pad_row["pad"]
        stack, shifted = stack_rsam_for_pad(
            rsam_frames,
            st,
            pad_row,
            inv_coords=inv_coords,
            velocity_mps=stack_velocity_mps,
            bin_s=stack_bin_s,
            normalize=normalize_before_stack,
        )
        stacks[pad] = stack
        if stack is None or stack.empty:
            print(f"  {pad}: no stack")
            continue

        stack_out = event_dir / f"stack_{safe_slug(pad)}.csv"
        shifted_out = event_dir / f"shifted_rsam_{safe_slug(pad)}.csv"
        stack.to_csv(stack_out, index=False)
        if shifted is not None:
            shifted.to_csv(shifted_out, index=False)
        print(f"  {pad}: stack samples={len(stack)}, channels={int(stack['n_seed_ids'].max())}; wrote {stack_out.name}")

        st_pad = stack_dataframe_to_stream(stack, pad=pad, event_id=event_id)
        stack_stream += st_pad
        det = detect_on_stack_stream(st_pad, event_dir, pad)
        if not det.empty:
            det["event_id"] = event_id
            det["slug"] = slug
            det["event_dir"] = str(event_dir)
            detections.append(det)
        plot_stack_detections(stack, det, event_dir, event_id, pad)

    # Save combined stack stream as MiniSEED for external inspection.
    if len(stack_stream) > 0:
        mseed_out = event_dir / "rsam_stack_traces.mseed"
        stack_stream.write(str(mseed_out), format="MSEED")
        print(f"  wrote {mseed_out}")

    # Plot all pad stacks together.
    launch_time = None
    for col in ["launch_time", "datetime", "t0", "event_time", "window_start"]:
        if col in row.index and pd.notna(row[col]):
            try:
                launch_time = ensure_utc(row[col])
                break
            except Exception:
                pass
    plot_stacks(stacks, event_dir, event_id, launch_time=launch_time)

    if detections:
        det_df = pd.concat(detections, ignore_index=True)
    else:
        det_df = pd.DataFrame()
    det_out = event_dir / "stack_detections.csv"
    det_df.to_csv(det_out, index=False)
    print(f"  detections: {len(det_df)}; wrote {det_out}")

    return {
        "event_id": event_id,
        "slug": slug,
        "status": "ok",
        "event_dir": str(event_dir),
        "n_raw_traces": len(st),
        "n_rsam_traces": len(rsam_frames),
        "n_pads": len(pads_to_try),
        "n_detections": len(det_df),
        "metadata_pad_column": pad_col,
        "metadata_pad_norm": pad_norm,
    }

## 8. Run the two-event prototype

This is deliberately small. Once the outputs look right, set `test_event_indices = None` or choose a longer list in the configuration cell.

In [ ]:
run_summaries = []
for i, row in event_df.iterrows():
    summary = process_one_event(row, sdsobject, run_index=i + 1)
    run_summaries.append(summary)

run_summary_df = pd.DataFrame(run_summaries)
summary_out = outdir / "run_summary.csv"
run_summary_df.to_csv(summary_out, index=False)
print(f"Wrote {summary_out}")
display(run_summary_df)

## 9. Combine detections across processed events

In [ ]:
all_det = []
for event_dir in run_summary_df.get("event_dir", []):
    p = Path(event_dir) / "stack_detections.csv"
    if p.exists():
        d = pd.read_csv(p)
        if not d.empty:
            all_det.append(d)

if all_det:
    all_detections_df = pd.concat(all_det, ignore_index=True)
else:
    all_detections_df = pd.DataFrame()

all_det_out = outdir / "all_stack_detections.csv"
all_detections_df.to_csv(all_det_out, index=False)
print(f"Wrote {all_det_out}")
display(all_detections_df.head(50))

## 10. Notes for scaling up

- The detector currently runs on **source-time RSAM stacks**, not corrected high-rate waveforms.
- Once stack detections look sensible, a later notebook/cell should reread tight windows around detections and compute physical metrics after response correction.
- For known launchpad rows, only that pad is stacked. If the metadata lacks a pad or the pad name does not match `launchpads.csv`, all pads are stacked.
- Sonic booms are not expected to be well represented by a fixed-launchpad stack. They probably need a separate single-station or trajectory-informed detector.